# plantid: fine-tune small CNNs on the 87-species working set (Colab A100)

This notebook trains **three** versions of the same small backbone (`mobilenet_v3_small`, ~2.5M params / ~10MB) on the 87-species `{leaf, bark, flower}` working set, and compares them **in-notebook** before anything is downloaded:

1. **CE-only**: standard cross-entropy fine-tuning (the baseline).
2. **CE + SupCon**: cross-entropy plus a supervised contrastive loss (Khosla et al. 2020) on a projection head, using two augmented views per image. The deployed matcher does k-NN over embeddings, not softmax classification, so SupCon directly optimizes for same-species embeddings clustering tightly - which is what k-NN actually needs.
3. **CE + edge channel**: same CE objective, but the input gets a 4th channel (a Canny edge map of the image) and a less aggressive crop (`scale=(0.8, 1.0)` vs `(0.6, 1.0)`), so the network has direct access to leaf/bark/flower outline shape (margin serration, lobing) in addition to color/texture. Motivated by qualitative error analysis (Step 11/12) on a previous run: several leaf mismatches had visually distinct outlines (e.g. *Chaerophyllum temulum*'s deeply dissected leaf vs *Aegopodium podagraria*'s simple toothed leaflets) that the CE/SupCon embeddings didn't separate.

**Previous run's results** (no edge channel; full writeup in `CNN_FINDINGS.md`): classifier-head top1 jumped from the classical-descriptor baseline (leaf 0.065, bark 0.098, flower 0.156) to leaf 0.58, bark 0.44, flower 0.72-0.74, and fusing all three organs' k-NN rankings now **beats** the best single organ by 12-16pp at top5/top10 (vs. hurting it before). CE and SupCon were essentially tied. This run adds the edge-channel variant as a third point of comparison.

**Comparison performed in-notebook**: a from-scratch k-NN matcher (mirrors `plantid/matching/classical.py`: train-split gallery minus flagged outliers, z-score standardized, inverse-distance-weighted k=15 voting) is run per-organ on `ce_emb`, `supcon_emb`, `supcon_proj`, and `ce_edge_emb`, reported alongside the classical-descriptor baselines:

| organ  | top1  | top5  | top10 |
|--------|-------|-------|-------|
| leaf   | 0.065 | 0.183 | 0.283 |
| bark   | 0.098 | 0.257 | 0.383 |
| flower | 0.156 | 0.348 | 0.472 |

**What gets downloaded** (`cnn_export.zip`): per-organ embeddings for all four variants (`descriptors_{organ}_{ce_emb,supcon_emb,supcon_proj,ce_edge_emb}.npz`, store.py-compatible), three model checkpoints + ONNX exports, and `metadata.json` with the full classifier-head and k-NN comparison tables.


In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## Step 1: upload the dataset manifest

Upload two small files from your local `plantid/data/processed/`:
- `plantnet_index.parquet` (~900KB) - image_id/species_id/organ/split manifest
- `outlier_scores.parquet` (~650KB) - Phase 2 label-noise flags

Images themselves (832MB) are **not** uploaded - the next step re-downloads them directly from PlantNet's CDN.

In [ ]:
from google.colab import files
print('Select plantnet_index.parquet and outlier_scores.parquet')
uploaded = files.upload()

In [ ]:
import json
import cv2
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.spatial.distance import cdist
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

ROOT = Path('/content/plantid_data')
IMAGES_DIR = ROOT / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

index = pd.read_parquet('plantnet_index.parquet')
outliers = pd.read_parquet('outlier_scores.parquet')
print(index.shape, index['species_id'].nunique(), 'species')
print(index.groupby('split').size())

## Step 2: download images from PlantNet

Threaded download with a pooled `requests.Session` per worker thread (plain `requests.get()` re-handshakes TCP+TLS on every call, which dominates latency under high concurrency to a remote CDN). Resumable - already-downloaded files are skipped via `dest.exists()`.

In [ ]:
IMAGE_URL = 'https://bs.plantnet.org/image/m/{image_id}'
MAX_WORKERS = 64

_thread_local = threading.local()


def _session():
    sess = getattr(_thread_local, 'session', None)
    if sess is None:
        sess = requests.Session()
        adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
        sess.mount('https://', adapter)
        _thread_local.session = sess
    return sess


def fetch(row):
    dest = IMAGES_DIR / row.species_id / row.organ / f'{row.image_id}.jpg'
    if dest.exists():
        return row.image_id, str(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    sess = _session()
    for attempt in range(3):
        try:
            resp = sess.get(IMAGE_URL.format(image_id=row.image_id), timeout=10)
            resp.raise_for_status()
            dest.write_bytes(resp.content)
            return row.image_id, str(dest)
        except requests.RequestException:
            time.sleep(0.5 * (attempt + 1))
    return row.image_id, None


paths = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(fetch, row) for row in index.itertuples()]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        image_id, path = fut.result()
        paths[image_id] = path

index['abs_path'] = index['image_id'].map(paths)
n_failed = index['abs_path'].isna().sum()
print(f'failed downloads: {n_failed} / {len(index)}')
index = index[index['abs_path'].notna()].reset_index(drop=True)

## Step 3: labels and outlier exclusion

Same convention as `plantid/matching/classical.py`: train on the `train` split with Phase-2 outlier-flagged images excluded; `val`/`test` untouched.

In [ ]:
species_ids = sorted(index['species_id'].unique())
species_to_idx = {sp: i for i, sp in enumerate(species_ids)}
species_to_name = dict(zip(index['species_id'], index['species_name']))
index['label'] = index['species_id'].map(species_to_idx)

flagged = set(outliers.loc[outliers['is_outlier'], 'image_id'])
is_train = index['split'] == 'train'
is_clean = ~index['image_id'].isin(flagged)
index['use_for_train'] = is_train & is_clean

print('train (clean):', index['use_for_train'].sum(), '/ train (all):', is_train.sum())
print('val:', (index['split'] == 'val').sum(), 'test:', (index['split'] == 'test').sum())

## Step 4: dataset and transforms

`PlantDataset` has a `two_view` mode: when `True`, `__getitem__` applies the train transform *twice independently* to the same image, returning `(view1, view2, label)` - the positive pair SupCon needs. The CE-only run uses `two_view=False`.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224
BATCH_SIZE = 64
EDGE_LOW, EDGE_HIGH = 50, 150  # Canny thresholds for the edge-channel variant

to_tensor = transforms.ToTensor()
normalize_rgb = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

# geometric-only transforms (applied to the PIL image, before tensor conversion
# so the edge map can be computed from the same augmented view as the RGB)
train_geom = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
])

# wider crop for the edge-channel variant, so leaf/organ outlines stay in frame
train_geom_wide = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
])

eval_geom = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
])


def compute_edge_channel(rgb: torch.Tensor) -> torch.Tensor:
    """Canny edge map of an (3,H,W) [0,1] tensor, returned as (1,H,W) in [-1, 1]."""
    gray = (rgb.mean(dim=0).numpy() * 255).astype(np.uint8)
    edges = cv2.Canny(gray, EDGE_LOW, EDGE_HIGH).astype(np.float32) / 255.0
    return (torch.from_numpy(edges).unsqueeze(0) - 0.5) / 0.5


def make_tensor(pil_img, geom, use_edge):
    pil_img = geom(pil_img)
    rgb = to_tensor(pil_img)
    edge = compute_edge_channel(rgb) if use_edge else None
    rgb = normalize_rgb(rgb)
    return torch.cat([rgb, edge], dim=0) if use_edge else rgb


class PlantDataset(Dataset):
    def __init__(self, df, geom, two_view=False, use_edge=False):
        self.df = df.reset_index(drop=True)
        self.geom = geom
        self.two_view = two_view
        self.use_edge = use_edge

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['abs_path']).convert('RGB')
        if self.two_view:
            return (make_tensor(img, self.geom, self.use_edge),
                    make_tensor(img, self.geom, self.use_edge),
                    row['label'])
        return make_tensor(img, self.geom, self.use_edge), row['label']


train_df = index[index['use_for_train']]
val_df = index[index['split'] == 'val']
test_df = index[index['split'] == 'test']

val_dl = DataLoader(PlantDataset(val_df, eval_geom), batch_size=BATCH_SIZE,
                     shuffle=False, num_workers=4, pin_memory=True)
test_dl = DataLoader(PlantDataset(test_df, eval_geom), batch_size=BATCH_SIZE,
                      shuffle=False, num_workers=4, pin_memory=True)

# 4-channel (RGB + edge) variants, for the ce_edge model
val_dl_edge = DataLoader(PlantDataset(val_df, eval_geom, use_edge=True), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4, pin_memory=True)
test_dl_edge = DataLoader(PlantDataset(test_df, eval_geom, use_edge=True), batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=4, pin_memory=True)

print('train/val/test:', len(train_df), len(val_df), len(test_df))


## Step 5: model and SupCon loss

`EmbeddingClassifier` exposes three things from one forward pass: classification `logits`, the 1280-dim penultimate `embedding`, and an L2-normalized 128-dim `projection` (used only by the SupCon loss; meaningless/untrained for the CE-only model). `SupConLoss` is the standard `L_out` formulation (Khosla et al. 2020) - for each anchor in the 2N-sample batch (N images x 2 views), all other samples sharing its label are positives.

In [ ]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, n_classes, proj_dim=128, in_channels=3):
        super().__init__()
        backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        if in_channels != 3:
            # adapt the stem conv to extra input channels (e.g. an edge map);
            # copy the pretrained RGB weights and init new channels from their mean
            old_conv = backbone.features[0][0]
            new_conv = nn.Conv2d(
                in_channels, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                stride=old_conv.stride, padding=old_conv.padding, bias=old_conv.bias is not None,
            )
            with torch.no_grad():
                new_conv.weight[:, :3] = old_conv.weight
                new_conv.weight[:, 3:] = old_conv.weight.mean(dim=1, keepdim=True)
            backbone.features[0][0] = new_conv
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        # classifier[0:2] = Linear(576, 1024) + Hardswish -> the 'embedding'
        self.embed = nn.Sequential(backbone.classifier[0], backbone.classifier[1])
        self.embed_dim = backbone.classifier[0].out_features
        self.dropout = backbone.classifier[2]
        self.classifier = nn.Linear(self.embed_dim, n_classes)
        self.projection = nn.Sequential(
            nn.Linear(self.embed_dim, self.embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(self.embed_dim, proj_dim),
        )

    def forward(self, x):
        f = self.features(x)
        f = self.avgpool(f)
        f = torch.flatten(f, 1)
        emb = self.embed(f)
        logits = self.classifier(self.dropout(emb))
        proj = F.normalize(self.projection(emb), dim=1)
        return logits, emb, proj


class SupConLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: (bsz, n_views, dim), L2-normalized. labels: (bsz,)
        device = features.device
        bsz, n_views, dim = features.shape
        features = features.reshape(bsz * n_views, dim)
        labels = labels.repeat(n_views)

        sim = torch.matmul(features, features.T) / self.temperature
        sim = sim - sim.max(dim=1, keepdim=True).values.detach()

        logits_mask = ~torch.eye(bsz * n_views, dtype=torch.bool, device=device)
        positive_mask = (labels.unsqueeze(0) == labels.unsqueeze(1)) & logits_mask

        exp_sim = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True))

        n_positives = positive_mask.sum(dim=1).clamp(min=1)
        mean_log_prob_pos = (positive_mask * log_prob).sum(dim=1) / n_positives
        return -mean_log_prob_pos.mean()


N_CLASSES = len(species_ids)
device = 'cuda' if torch.cuda.is_available() else 'cpu'


## Step 6: training function

`run_training(use_supcon, ...)` builds a fresh model (same seed -> same init across runs), trains for `epochs`, and returns the best-val-accuracy checkpoint. With `use_supcon=True`, each step does two forward passes (one per view) and `loss = CE(logits_view1, y) + LAMBDA_SUPCON * SupCon([proj1, proj2], y)`.

New params support the edge-channel variant: `in_channels` (3 or 4), `geom` (train-time geometric transform, defaults to `train_geom`), `use_edge` (whether `PlantDataset` appends the Canny edge channel), and `val_dl` (which validation loader to track best-val-accuracy against - 4-channel models need `val_dl_edge`).


In [ ]:
EPOCHS = 15
LR = 3e-4
LAMBDA_SUPCON = 1.0
SEED = 42


def run_training(use_supcon, epochs=EPOCHS, lr=LR, lambda_supcon=LAMBDA_SUPCON, seed=SEED,
                  in_channels=3, geom=None, use_edge=False, val_dl=val_dl):
    torch.manual_seed(seed)
    model = EmbeddingClassifier(N_CLASSES, in_channels=in_channels).to(device)

    train_dl = DataLoader(
        PlantDataset(train_df, geom or train_geom, two_view=use_supcon, use_edge=use_edge),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True,
    )

    ce_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    supcon_criterion = SupConLoss(temperature=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()

    def train_epoch():
        model.train()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for batch in train_dl:
            if use_supcon:
                x1, x2, y = batch
                x = torch.cat([x1, x2], dim=0).to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                with torch.cuda.amp.autocast():
                    logits, emb, proj = model(x)
                    logits1, _ = logits.chunk(2, dim=0)
                    proj1, proj2 = proj.chunk(2, dim=0)
                    loss = ce_criterion(logits1, y) + lambda_supcon * supcon_criterion(
                        torch.stack([proj1, proj2], dim=1), y
                    )
                preds = logits1
            else:
                x, y = batch
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with torch.cuda.amp.autocast():
                    logits, emb, proj = model(x)
                    loss = ce_criterion(logits, y)
                preds = logits
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * y.size(0)
            total_correct += (preds.argmax(1) == y).sum().item()
            total_n += y.size(0)
        return total_loss / total_n, total_correct / total_n

    @torch.no_grad()
    def eval_epoch(dl):
        model.eval()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                logits, emb, proj = model(x)
                loss = ce_criterion(logits, y)
            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n += y.size(0)
        return total_loss / total_n, total_correct / total_n

    tag = 'supcon' if use_supcon else 'ce'
    best_val_acc, best_state = 0.0, None
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch()
        val_loss, val_acc = eval_epoch(val_dl)
        scheduler.step()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'[{tag}] epoch {epoch + 1:2d}/{epochs}  train_loss={train_loss:.3f} train_acc={train_acc:.3f}  '
              f'val_loss={val_loss:.3f} val_acc={val_acc:.3f}  best_val_acc={best_val_acc:.3f}')

    model.load_state_dict(best_state)
    return model, best_val_acc


## Step 7: train all three models

~10 min for the CE-only run and ~20 min for the SupCon run on an A100 (roughly 2x cost from the two-view forward pass). The CE+edge run (Step 7c) is single-view like CE-only, so also ~10 min - total ~40 min.


In [ ]:
model_ce, val_acc_ce = run_training(use_supcon=False)
print('CE best val acc:', val_acc_ce)

In [ ]:
model_supcon, val_acc_supcon = run_training(use_supcon=True)
print('SupCon best val acc:', val_acc_supcon)

In [ ]:
model_ce_edge, val_acc_ce_edge = run_training(
    use_supcon=False, in_channels=4, use_edge=True, geom=train_geom_wide, val_dl=val_dl_edge,
)
print('CE+edge best val acc:', val_acc_ce_edge)


## Step 8: test-set top-1/5/10 (classifier head, overall and per-organ)

In [ ]:
TOP_KS = (1, 5, 10)


@torch.no_grad()
def topk_accuracy(model, dl, df):
    model.eval()
    all_logits = []
    for x, _ in dl:
        x = x.to(device)
        with torch.cuda.amp.autocast():
            logits, _, _ = model(x)
        all_logits.append(logits.float().cpu())
    logits = torch.cat(all_logits)
    labels = torch.tensor(df['label'].values)
    topk = logits.topk(max(TOP_KS), dim=1).indices
    return {f'top{k}': (topk[:, :k] == labels[:, None]).any(1).float().mean().item() for k in TOP_KS}


MODELS = {
    'ce': (model_ce, eval_geom, False, test_dl),
    'supcon': (model_supcon, eval_geom, False, test_dl),
    'ce_edge': (model_ce_edge, eval_geom, True, test_dl_edge),
}

classifier_results = {}
for name, (model, geom, use_edge, dl) in MODELS.items():
    classifier_results[name] = {'overall': topk_accuracy(model, dl, test_df)}
    for organ in ('leaf', 'bark', 'flower'):
        sub_df = test_df[test_df['organ'] == organ]
        sub_dl = DataLoader(PlantDataset(sub_df, geom, use_edge=use_edge), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=2)
        classifier_results[name][organ] = topk_accuracy(model, sub_dl, sub_df)
    print(name, json.dumps(classifier_results[name], indent=2))


## Step 9: extract embeddings for all images

In [ ]:
@torch.no_grad()
def embed_all(model, dl):
    model.eval()
    embs, projs = [], []
    for x, _ in dl:
        x = x.to(device)
        with torch.cuda.amp.autocast():
            _, emb, proj = model(x)
        embs.append(emb.float().cpu().numpy())
        projs.append(proj.float().cpu().numpy())
    return np.concatenate(embs), np.concatenate(projs)


all_dl = DataLoader(PlantDataset(index, eval_geom), batch_size=BATCH_SIZE,
                     shuffle=False, num_workers=4)
all_dl_edge = DataLoader(PlantDataset(index, eval_geom, use_edge=True), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4)

ce_emb, _ = embed_all(model_ce, all_dl)
supcon_emb, supcon_proj = embed_all(model_supcon, all_dl)
ce_edge_emb, _ = embed_all(model_ce_edge, all_dl_edge)
print('ce_emb', ce_emb.shape, 'supcon_emb', supcon_emb.shape,
      'supcon_proj', supcon_proj.shape, 'ce_edge_emb', ce_edge_emb.shape)


## Step 10: in-notebook k-NN comparison

A from-scratch k-NN matcher mirroring `plantid/matching/classical.py`: per organ, gallery = train split minus flagged outliers, z-score standardized (skipped for `supcon_proj`, which is already L2-normalized - Euclidean distance on normalized vectors is monotonic in cosine similarity), `k=15` inverse-distance-weighted voting, top-1/5/10 on the test split.

In [ ]:
K_MATCH = 15


def knn_topk(embeddings, organ, standardize=True, k=K_MATCH, top_ks=TOP_KS):
    sub = index[index['organ'] == organ].reset_index(drop=True)
    emb = embeddings[(index['organ'] == organ).values]

    train_mask = (sub['split'] == 'train').values & sub['use_for_train'].values
    test_mask = (sub['split'] == 'test').values

    gallery = emb[train_mask]
    gallery_labels = sub['species_id'].values[train_mask]
    query = emb[test_mask]
    query_labels = sub['species_id'].values[test_mask]

    if standardize:
        mean, std = gallery.mean(0), gallery.std(0)
        std[std < 1e-12] = 1.0
        gallery = (gallery - mean) / std
        query = (query - mean) / std

    dists = cdist(query, gallery)
    nn_idx = np.argsort(dists, axis=1)[:, :k]

    hits = {tk: 0 for tk in top_ks}
    for qi, neighbors in enumerate(nn_idx):
        weights = 1.0 / (dists[qi, neighbors] + 1e-6)
        votes = {}
        for sp, w in zip(gallery_labels[neighbors], weights):
            votes[sp] = votes.get(sp, 0.0) + w
        ranked_sp = [sp for sp, _ in sorted(votes.items(), key=lambda x: -x[1])]
        for tk in top_ks:
            if query_labels[qi] in ranked_sp[:tk]:
                hits[tk] += 1

    n = test_mask.sum()
    return {f'top{tk}': hits[tk] / n for tk in top_ks}


CLASSICAL_BASELINE = {
    'leaf': {'top1': 0.065, 'top5': 0.183, 'top10': 0.283},
    'bark': {'top1': 0.098, 'top5': 0.257, 'top10': 0.383},
    'flower': {'top1': 0.156, 'top5': 0.348, 'top10': 0.472},
}

knn_results = {}
variants = {
    'classical': None,  # filled from CLASSICAL_BASELINE below
    'ce_emb': (ce_emb, True),
    'supcon_emb': (supcon_emb, True),
    'supcon_proj': (supcon_proj, False),
    'ce_edge_emb': (ce_edge_emb, True),
}

print(f"{'organ':<8s} {'variant':<12s} {'top1':>6s} {'top5':>6s} {'top10':>6s}")
for organ in ('leaf', 'bark', 'flower'):
    print(f"{organ:<8s} {'classical':<12s} " +
          ' '.join(f'{CLASSICAL_BASELINE[organ][f"top{tk}"]:6.3f}' for tk in TOP_KS))
    for name, spec in variants.items():
        if spec is None:
            continue
        emb, standardize = spec
        res = knn_topk(emb, organ, standardize=standardize)
        knn_results[(organ, name)] = res
        print(f"{'':<8s} {name:<12s} " + ' '.join(f'{res[f"top{tk}"]:6.3f}' for tk in TOP_KS))
    print()


## Step 11: qualitative inspection — classifier successes and failures

Build intuition about *why* the model gets things right or wrong before exporting anything. For each organ, shows a grid of correctly classified test images and a grid of misclassified ones, each titled with the true species and the model's top-3 predictions (with probabilities).

Things to look for:
- **Incorrect grids**: are the top-3 predictions visually/taxonomically plausible confusions (e.g. two species in the same genus), or is the model confidently wrong on something unrelated (often a sign of a mislabeled or low-quality image — see Phase 2's `outlier_scores.parquet`)?
- **Correct grids**: do the "easy" images share some property (centered subject, clean background, good lighting) that's missing from the failures?

Set `INSPECT_MODEL` to `'ce'`, `'supcon'`, or `'ce_edge'` to inspect a different model (the `ce_edge` model uses 4-channel input, so its predictions come from `test_dl_edge`).


In [ ]:
import matplotlib.pyplot as plt

idx_to_species = {i: sp for sp, i in species_to_idx.items()}
test_df_r = test_df.reset_index(drop=True)


@torch.no_grad()
def predict_topk(model, dl, k=5):
    model.eval()
    all_probs = []
    for x, _ in dl:
        x = x.to(device)
        with torch.cuda.amp.autocast():
            logits, _, _ = model(x)
        all_probs.append(F.softmax(logits.float(), dim=1).cpu())
    return torch.cat(all_probs).topk(k, dim=1)


INSPECT_MODEL = 'supcon'  # 'ce', 'supcon', or 'ce_edge'
inspect_model, inspect_dl = MODELS[INSPECT_MODEL][0], MODELS[INSPECT_MODEL][3]
topk_probs, topk_idx = predict_topk(inspect_model, inspect_dl, k=5)
topk_probs, topk_idx = topk_probs.numpy(), topk_idx.numpy()
correct = topk_idx[:, 0] == test_df_r['label'].values


def show_grid(mask, organ, n=8, title=''):
    candidates = np.where(mask & (test_df_r['organ'] == organ).values)[0]
    if len(candidates) == 0:
        print(f'no examples: {title}')
        return
    chosen = np.random.choice(candidates, size=min(n, len(candidates)), replace=False)
    ncols = 4
    nrows = -(-len(chosen) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, i in zip(axes, chosen):
        row = test_df_r.iloc[i]
        ax.imshow(Image.open(row['abs_path']).convert('RGB'))
        true_name = species_to_name[row['species_id']]
        preds = [(species_to_name[idx_to_species[j]], p) for j, p in zip(topk_idx[i, :3], topk_probs[i, :3])]
        pred_str = '\n'.join(f'{n} ({p:.2f})' for n, p in preds)
        ax.set_title(f'true: {true_name}\n{pred_str}', fontsize=8)
        ax.axis('off')
    for ax in axes[len(chosen):]:
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


for organ in ('leaf', 'bark', 'flower'):
    organ_acc = correct[(test_df_r['organ'] == organ).values].mean()
    print(f'=== {organ}: {INSPECT_MODEL} classifier top-1 test accuracy = {organ_acc:.3f} ===')
    show_grid(correct, organ, title=f'{organ}: correct (top-3 shown)')
    show_grid(~correct, organ, title=f'{organ}: INCORRECT (top-3 shown)')


## Step 12: qualitative inspection — k-NN retrieval (query + nearest neighbors)

This is closer to the actual deployed matcher (`matching/classical.py` is k-NN over descriptors). For a few correct and incorrect test queries per organ, shows the query image alongside its top-5 retrieved gallery neighbors and their species/distance — useful for spotting whether failures come from genuinely confusable species, a noisy/mislabeled gallery image among the neighbors, or an embedding that just isn't discriminative for that organ.

`KNN_INSPECT_VARIANT`/`KNN_INSPECT_STANDARDIZE` default to the variants that looked best in Step 10's table — adjust per organ based on what you saw there. `ce_edge_emb` is also a valid choice; compare its neighbors against `ce_emb`/`supcon_emb` for the leaf-margin failure cases noted in Step 11 to see whether the edge channel changed what gets retrieved.


In [ ]:
def knn_visual_examples(embeddings, organ, standardize=True, k=5, n_correct=2, n_incorrect=2, seed=0):
    rng = np.random.default_rng(seed)
    sub = index[index['organ'] == organ].reset_index(drop=True)
    emb = embeddings[(index['organ'] == organ).values]

    train_mask = (sub['split'] == 'train').values & sub['use_for_train'].values
    test_mask = (sub['split'] == 'test').values

    gallery = emb[train_mask]
    gallery_labels = sub['species_id'].values[train_mask]
    gallery_paths = sub['abs_path'].values[train_mask]
    query = emb[test_mask]
    query_labels = sub['species_id'].values[test_mask]
    query_paths = sub['abs_path'].values[test_mask]

    if standardize:
        mean, std = gallery.mean(0), gallery.std(0)
        std[std < 1e-12] = 1.0
        gallery_n = (gallery - mean) / std
        query_n = (query - mean) / std
    else:
        gallery_n, query_n = gallery, query

    dists = cdist(query_n, gallery_n)
    nn_idx = np.argsort(dists, axis=1)[:, :k]

    pred_sp = []
    for qi, neighbors in enumerate(nn_idx):
        w = 1.0 / (dists[qi, neighbors] + 1e-6)
        votes = {}
        for sp, wi in zip(gallery_labels[neighbors], w):
            votes[sp] = votes.get(sp, 0.0) + wi
        pred_sp.append(max(votes.items(), key=lambda x: x[1])[0])
    pred_sp = np.array(pred_sp)
    correct = pred_sp == query_labels

    for label, mask, n in (('CORRECT', correct, n_correct), ('INCORRECT', ~correct, n_incorrect)):
        candidates = np.where(mask)[0]
        if len(candidates) == 0:
            continue
        for qi in rng.choice(candidates, size=min(n, len(candidates)), replace=False):
            fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 3.2))
            axes[0].imshow(Image.open(query_paths[qi]).convert('RGB'))
            axes[0].set_title(f'QUERY ({label})\ntrue: {species_to_name[query_labels[qi]]}\n'
                               f'pred: {species_to_name[pred_sp[qi]]}', fontsize=8)
            axes[0].axis('off')
            for j, ni in enumerate(nn_idx[qi]):
                axes[j + 1].imshow(Image.open(gallery_paths[ni]).convert('RGB'))
                axes[j + 1].set_title(f'NN{j+1} d={dists[qi, ni]:.1f}\n{species_to_name[gallery_labels[ni]]}', fontsize=8)
                axes[j + 1].axis('off')
            plt.tight_layout()
            plt.show()


# defaults below; adjust per organ based on Step 10's printed comparison
KNN_INSPECT_VARIANT = {'leaf': supcon_emb, 'bark': supcon_emb, 'flower': supcon_proj}
KNN_INSPECT_STANDARDIZE = {'leaf': True, 'bark': True, 'flower': False}

for organ in ('leaf', 'bark', 'flower'):
    print(f'=== {organ}: k-NN retrieval examples ===')
    knn_visual_examples(KNN_INSPECT_VARIANT[organ], organ, standardize=KNN_INSPECT_STANDARDIZE[organ])


## Step 13: export results

Embeddings for all three variants (per organ), both model checkpoints + ONNX exports, and `metadata.json` with the full classifier-head and k-NN comparison tables.

In [ ]:
OUT = Path('/content/cnn_export')
OUT.mkdir(exist_ok=True)

EMBEDDING_VARIANTS = {
    'ce_emb': ce_emb,
    'supcon_emb': supcon_emb,
    'supcon_proj': supcon_proj,
    'ce_edge_emb': ce_edge_emb,
}

for organ in ('leaf', 'bark', 'flower'):
    mask = (index['organ'] == organ).values
    base = {
        'image_id': index['image_id'].values[mask],
        'species_id': index['species_id'].values[mask],
        'species_name': index['species_name'].values[mask],
        'split': index['split'].values[mask],
    }
    for variant_name, emb in EMBEDDING_VARIANTS.items():
        np.savez_compressed(OUT / f'descriptors_{organ}_{variant_name}.npz', descriptor=emb[mask], **base)
    print(organ, 'exported', list(EMBEDDING_VARIANTS.keys()))


In [ ]:
torch.save(model_ce.state_dict(), OUT / 'mobilenet_v3_small_ce.pt')
torch.save(model_supcon.state_dict(), OUT / 'mobilenet_v3_small_supcon.pt')
torch.save(model_ce_edge.state_dict(), OUT / 'mobilenet_v3_small_ce_edge.pt')

EXPORT_MODELS = {
    'ce': (model_ce, 3),
    'supcon': (model_supcon, 3),
    'ce_edge': (model_ce_edge, 4),
}
for name, (model, in_channels) in EXPORT_MODELS.items():
    dummy = torch.randn(1, in_channels, IMG_SIZE, IMG_SIZE, device=device)
    torch.onnx.export(
        model, dummy, str(OUT / f'mobilenet_v3_small_{name}.onnx'),
        input_names=['image'], output_names=['logits', 'embedding', 'projection'],
        dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'},
                       'embedding': {0: 'batch'}, 'projection': {0: 'batch'}},
        opset_version=17, dynamo=False,
    )

metadata = {
    'species_ids': species_ids,
    'species_names': [species_to_name[s] for s in species_ids],
    'classifier_head': classifier_results,
    'knn_comparison': {f'{organ}__{name}': res for (organ, name), res in knn_results.items()},
    'classical_baseline': CLASSICAL_BASELINE,
    'best_val_acc': {'ce': val_acc_ce, 'supcon': val_acc_supcon, 'ce_edge': val_acc_ce_edge},
    'embed_dim': int(ce_emb.shape[1]),
    'proj_dim': int(supcon_proj.shape[1]),
    'epochs': EPOCHS,
    'lambda_supcon': LAMBDA_SUPCON,
    'img_size': IMG_SIZE,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
    'edge_channel': {'canny_thresholds': [EDGE_LOW, EDGE_HIGH], 'crop_scale': [0.8, 1.0]},
}
with open(OUT / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata['knn_comparison'], indent=2))


In [ ]:
import shutil

shutil.make_archive('/content/cnn_export', 'zip', OUT)
files.download('/content/cnn_export.zip')

## Next steps (back in the local `plantid` repo)

1. Extract `cnn_export.zip` into `data/processed/` - twelve `descriptors_{organ}_{ce_emb,supcon_emb,supcon_proj,ce_edge_emb}.npz` files land alongside the existing classical `descriptors_{organ}.npz`.
2. The `knn_comparison` table in `metadata.json` already tells you which variant won in-notebook (k=15, fixed). Add the winning variant(s) as a new descriptor type in `features/store.py` / `matching/classical.py` (z-score for `*_emb`, cosine/no-standardization for `supcon_proj`) and re-run `eval/match_eval.py` with its k-sweep, plus `matching/fusion.py`, for an apples-to-apples comparison with the classical pipeline's val-tuned k.
3. If `ce_edge` wins on leaf (or elsewhere), `mobilenet_v3_small_ce_edge.onnx` takes a 4-channel input - any inference code needs to compute the same Canny edge channel (`EDGE_LOW`/`EDGE_HIGH` in `metadata.json`) and concatenate it before normalization.
4. (Future work, not in this notebook) If `ce_edge` doesn't fully resolve the outline-shape confusions seen in Step 11/12, the next lever is better foreground isolation: a pretrained segmentation model (e.g. SAM) prompted near the image center could crop/mask the leaf/bark/flower out of the cluttered background before feeding the CNN, and/or before re-trying `imret`'s ORB+FAISS matcher (Phase 3 found it near-chance on whole, cluttered images - isolating the organ removes most of the keypoint noise that caused that).
